In [1]:
%load_ext autoreload
%autoreload 2

import os
import joblib
import gc
import time
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
import sys

os.environ["TABPFN_TOKEN"] = "tabpfn_sk_s-vJ41-nt_-nHcf2dZrJ5DTBvU4qG7VGuPdwD4E807g"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

In [2]:
'''def load_and_split_data(train_path, test_path):
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    # Conserviamo gli ID per poter ricongiungere le predizioni dopo
    test_patient_ids = df_test['patient_id']
    test_operator_ids = df_test['operator_id']
    
    cols_to_drop = ['patient_id', 'operator_id', 'link_label', 'planning_date']
    
    X_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns])
    y_train = df_train['link_label']
    
    X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])
    y_test = df_test['link_label']
    
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe), Test ({X_test.shape[0]} righe)")
    return X_train, y_train, X_test, y_test, test_patient_ids, test_operator_ids'''


def load_and_split_data(train_path, test_path):
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    cols_to_drop = ['patient_id', 'operator_id', 'link_label', 'planning_date']
    
    X_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns])
    y_train = df_train['link_label']
    
    X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])
    y_test = df_test['link_label']
    
    return X_train, y_train, X_test, y_test

In [3]:
'''X_train, y_train, X_test, y_test, test_pat_ids, test_op_ids = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')

X_train_np = X_train.values.astype(np.float32)
X_test_np  = X_test.values.astype(np.float32)

print("--- Addestramento TabPFN Classifier per Link Prediction ---")
classifier = TabPFNClassifier(n_estimators=4)
classifier.fit(X_train_np, y_train.values)

# Estraiamo le probabilità per la classe 1 (Assegnazione Valida)
y_pred_proba = classifier.predict_proba(X_test_np)[:, 1]

# 1. Calcolo Affinità (Da 0 a 100)
affinity_scores = np.round(y_pred_proba * 100).astype(int)

# 2. Calcolo Confidenza (1 = Sicurissimo, 4 = Incertezza totale)
# Si basa sulla distanza dalla probabilità neutrale del 50% (0.50)
def get_confidence_level(prob):
    if prob >= 0.85 or prob <= 0.15: return 1
    if prob >= 0.70 or prob <= 0.30: return 2
    if prob >= 0.55 or prob <= 0.45: return 3
    return 4

conf_levels = np.array([get_confidence_level(p) for p in y_pred_proba])


df_results = pd.DataFrame({
    'Patient': test_pat_ids,
    'Operator': test_op_ids,
    'Probability': np.round(y_pred_proba, 3),
    'Affinity': affinity_scores,
    'Confidence_Level': conf_levels
})

print(df_results.head(10))

y_pred_class = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_class)
auc = roc_auc_score(y_test, y_pred_proba)
loss = log_loss(y_test, y_pred_proba)

print(f"Valutazione Modello -> Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f} | LogLoss: {loss:.4f}")

# Salviamo il modello
os.makedirs("saved_models", exist_ok=True)
joblib.dump(classifier, "saved_models/tabpfn_link_predictor.pkl")
joblib.dump(X_train.columns.tolist(), "saved_models/train_columns.pkl")
print("Modello salvato!")'''

'X_train, y_train, X_test, y_test, test_pat_ids, test_op_ids = load_and_split_data(\'data/train_dataset.csv\', \'data/test_dataset.csv\')\n\nX_train_np = X_train.values.astype(np.float32)\nX_test_np  = X_test.values.astype(np.float32)\n\nprint("--- Addestramento TabPFN Classifier per Link Prediction ---")\nclassifier = TabPFNClassifier(n_estimators=4)\nclassifier.fit(X_train_np, y_train.values)\n\n# Estraiamo le probabilità per la classe 1 (Assegnazione Valida)\ny_pred_proba = classifier.predict_proba(X_test_np)[:, 1]\n\n# 1. Calcolo Affinità (Da 0 a 100)\naffinity_scores = np.round(y_pred_proba * 100).astype(int)\n\n# 2. Calcolo Confidenza (1 = Sicurissimo, 4 = Incertezza totale)\n# Si basa sulla distanza dalla probabilità neutrale del 50% (0.50)\ndef get_confidence_level(prob):\n    if prob >= 0.85 or prob <= 0.15: return 1\n    if prob >= 0.70 or prob <= 0.30: return 2\n    if prob >= 0.55 or prob <= 0.45: return 3\n    return 4\n\nconf_levels = np.array([get_confidence_level(p) for

In [4]:
orgs_to_train = ["Org6", "Org9"]
os.makedirs("saved_models", exist_ok=True)

for org in orgs_to_train:
    train_path = f"data/train_dataset_{org}.csv"
    test_path = f"data/test_dataset_{org}.csv"
    
    if not os.path.exists(train_path) or not os.path.exists(test_path):
        print(f"\n[!] Dataset per {org} non trovati. Salto l'addestramento.")
        continue
        
    print(f" Addestramento Modello TabPFN per {org}\n")
    
    X_train, y_train, X_test, y_test = load_and_split_data(train_path, test_path)
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe), Test ({X_test.shape[0]} righe)")
    
    X_train_np = X_train.values.astype(np.float32)
    X_test_np  = X_test.values.astype(np.float32)
    
    classifier = TabPFNClassifier(n_estimators=4)
    classifier.fit(X_train_np, y_train.values)
    
    y_pred_proba = classifier.predict_proba(X_test_np)[:, 1]
    y_pred_bin = classifier.predict(X_test_np)
    
    auc = roc_auc_score(y_test, y_pred_proba)
    loss = log_loss(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_bin)
    
    print(f"Valutazione {org} -> Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f} | LogLoss: {loss:.4f}")
    
    # Salvataggio dinamico dei modelli e delle colonne
    joblib.dump(classifier, f"saved_models/tabpfn_link_predictor_{org}.pkl")
    joblib.dump(X_train.columns.tolist(), f"saved_models/train_columns_{org}.pkl")
    print(f"Modelli esportati con successo per {org}.")

 Addestramento Modello TabPFN per Org6

Dataset caricati: Train (16284 righe), Test (4071 righe)
Valutazione Org6 -> Accuracy: 0.9101 | ROC-AUC: 0.9298 | LogLoss: 0.2648
Modelli esportati con successo per Org6.
 Addestramento Modello TabPFN per Org9

Dataset caricati: Train (7942 righe), Test (1986 righe)
Valutazione Org9 -> Accuracy: 0.9033 | ROC-AUC: 0.9252 | LogLoss: 0.2766
Modelli esportati con successo per Org9.
